# Buổi E5: Voting và Stacking, ghép những mô hình khác loại

**Hiện trường.** Chị Hạnh có ba mô hình đã dựng ở các buổi trước: hồi quy logistic, rừng ngẫu nhiên và SVM. Mỗi cái giỏi một kiểu, sai một kiểu. Cô hỏi: *"ghép cả ba lại có hơn cái tốt nhất không? và ghép thế nào?"*

Hai cách ghép:

- **Voting** (bỏ phiếu): mỗi mô hình một phiếu, lấy số đông (hoặc trung bình xác suất).
- **Stacking** (xếp tầng): thêm một **mô hình thứ hai** học cách **tin ai khi nào** từ dự đoán của các mô hình tầng một.

**Ẩn dụ:** hội đồng ba bác sĩ khác chuyên khoa. Voting là biểu quyết. Stacking là có một **trưởng hội đồng** biết bác sĩ nào hay đúng với loại ca nào, rồi cân nhắc ý kiến họ.

**Mindset của người làm dữ liệu:** ghép chỉ có lợi nếu các thành viên **sai khác nhau**. Ba bản sao của một mô hình thì ghép cũng vô ích.

**Lưu ý:** dữ liệu sinh ra bằng `make_classification` (dựng có chủ đích để các mô hình mạnh yếu khác nhau).

## Bản đồ các chặng

| Chặng | Câu hỏi của chặng |
|---|---|
| Chặng 0 | Thấy ngay: ba mô hình riêng và bỏ phiếu |
| Chặng 1 | Bỏ phiếu cứng và mềm |
| Chặng 2 | Đa dạng: vì sao ba bản sao không đủ |
| Chặng 3 | Stacking: trưởng hội đồng học cách tin ai |
| Chặng 4 | Cái giá: thời gian và độ phức tạp |
| Chặng 5 | Mở tập thi một lần |

## Chặng 0: Thấy ngay, ba mô hình và bỏ phiếu

**Câu hỏi:** ghép ba mô hình khác loại có hơn mô hình tốt nhất?

Dữ liệu: 4000 dòng, 20 cột, mỗi nhóm gồm **3 cụm** (khiến ranh giới cong và các mô hình khác nhau cách học). Nhãn có 5% nhiễu.

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
BLUE, GRAY = "#1F4E79", "#8C8C8C"
plt.rcParams["figure.figsize"] = (6, 4)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

**Giải thích từng dòng**

- `import numpy as np`: mượn thư viện numpy (máy tính bỏ túi cho cả dãy số), đặt tên ngắn là `np` để gõ cho nhanh.
- `import pandas as pd`: mượn pandas, thư viện làm việc với bảng số liệu giống Excel, tên ngắn là `pd`.
- `import matplotlib.pyplot as plt`: mượn công cụ vẽ biểu đồ, tên ngắn là `plt`.
- `BLUE, GRAY = "#1F4E79", "#8C8C8C"`: gán hai màu dùng suốt bài (xanh đậm và xám). Một dòng gán cùng lúc hai biến, mỗi biến một mã màu.
- `plt.rcParams["figure.figsize"] = (6, 4)`: đặt cỡ hình mặc định: rộng 6, cao 4 (đơn vị inch).
- `plt.rcParams["axes.spines.top"] = False`: bỏ đường viền phía trên khung hình cho gọn mắt.
- `plt.rcParams["axes.spines.right"] = False`: bỏ luôn đường viền bên phải.

In [2]:
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_val_score
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, VotingClassifier
from sklearn.svm import SVC
from sklearn.calibration import CalibratedClassifierCV
X, y = make_classification(n_samples=4000, n_features=20, n_informative=8, n_redundant=2, n_clusters_per_class=3, flip_y=0.05, class_sep=1.0, random_state=0)
X_rest, X_test, y_rest, y_test = train_test_split(X, y, test_size=0.25, random_state=0, stratify=y)
cv = StratifiedKFold(5, shuffle=True, random_state=0)
members = {
    "logistic": make_pipeline(StandardScaler(), LogisticRegression(max_iter=2000)),
    "forest": RandomForestClassifier(n_estimators=200, random_state=0, n_jobs=-1),
    "svm": make_pipeline(StandardScaler(), CalibratedClassifierCV(SVC(C=3, random_state=0), ensemble=False)),
}
vote = VotingClassifier(list(members.items()), voting="hard")
rows = []
for name, m in {**members, "vote (hard)": vote}.items():
    s = cross_val_score(m, X_rest, y_rest, cv=cv, n_jobs=-1)
    rows.append({"model": name, "CV accuracy": s.mean(), "std": s.std()})
first = pd.DataFrame(rows)
print(first.round(3).to_string(index=False))

      model  CV accuracy   std
   logistic        0.851 0.015
     forest        0.894 0.018
        svm        0.901 0.017
vote (hard)        0.896 0.021


**Giải thích từng dòng**

- `X_rest, X_test, ...`: `X_test` niêm phong đến chặng 5; mọi so sánh dùng `X_rest` với kiểm định chéo.
- `members = {...}`: từ điển ba thành viên: tên → mô hình. Logistic và SVM nằm trong pipeline có chuẩn hóa.
- `CalibratedClassifierCV(SVC(C=3), ensemble=False)`: SVM tự nó không có xác suất; bọc trong bộ hiệu chỉnh để có predict_proba (đã học ở C6).
- `VotingClassifier(list(members.items()), voting="hard")`: bộ bỏ phiếu: nhận danh sách cặp (tên, mô hình). `voting="hard"` nghĩa là mỗi mô hình bỏ **một nhãn**, nhãn nhiều phiếu nhất thắng.
- `{**members, "vote (hard)": vote}`: gộp từ điển ba thành viên với thêm bộ bỏ phiếu để đo chung một vòng.

**Cách đọc.**
- **Con số:** logistic **85,1%**, rừng **89,4%**, SVM **90,1%**, bỏ phiếu **89,6%**.
- **Bức tranh:** so bỏ phiếu với **mô hình riêng tốt nhất** (90,1%), và nhớ độ lệch cỡ 1,8 điểm phần trăm: nếu chênh lệch nhỏ hơn độ lệch thì chưa thể nói bỏ phiếu hơn.
- **Câu hỏi tiếp:** "hard" bỏ qua việc mỗi mô hình **chắc chắn đến đâu**. Có cách nào dùng thông tin đó?

**Mang về chặng 0:** bỏ phiếu chỉ đáng khi vượt rõ mô hình riêng tốt nhất; so bằng kiểm định chéo, kèm độ lệch.

## Chặng 1: Bỏ phiếu cứng và mềm

**Câu hỏi:** "7 người nói có, 3 người nói không" khác gì "mỗi người nói chắc 51%"?

- **Cứng (hard):** mỗi mô hình một phiếu cho một nhãn.
- **Mềm (soft):** mỗi mô hình đưa **xác suất**; lấy **trung bình xác suất** rồi chọn nhãn cao nhất. Mô hình **chắc chắn** có ảnh hưởng lớn hơn mô hình lưỡng lự.

Mềm cần mô hình có `predict_proba`, và nên dùng xác suất **đáng tin** (D5).

In [3]:
soft = VotingClassifier(list(members.items()), voting="soft")
s_soft = cross_val_score(soft, X_rest, y_rest, cv=cv, n_jobs=-1)
print("hard vote:", round(first.loc[3, "CV accuracy"], 4), "| soft vote:", round(s_soft.mean(), 4), "| best single:", round(first.loc[:2, "CV accuracy"].max(), 4))

hard vote: 0.896 | soft vote: 0.897 | best single: 0.901


**Giải thích từng dòng**

- `voting="soft"`: bỏ phiếu mềm: trung bình `predict_proba` của các thành viên.

In [4]:
a, b, ya, yb = train_test_split(X_rest, y_rest, test_size=0.25, random_state=1, stratify=y_rest)
probs = {n: m.fit(a, ya).predict_proba(b)[:, 1] for n, m in members.items()}
avg = np.mean(list(probs.values()), axis=0)
print({n: round(float(((p > 0.5) == yb).mean()), 3) for n, p in probs.items()}, "| averaged probability:", round(float(((avg > 0.5) == yb).mean()), 3))

{'logistic': 0.831, 'forest': 0.897, 'svm': 0.897} | averaged probability: 0.884


**Giải thích từng dòng**

- `{n: m.fit(a, ya).predict_proba(b)[:, 1] for n, m in members.items()}`: với mỗi mô hình: học trên `a`, lấy xác suất nhãn 1 cho `b`. Đây chính là việc bên trong bỏ phiếu mềm.
- `np.mean(list(probs.values()), axis=0)`: trung bình ba mảng xác suất theo từng điểm.

**Cách đọc.**
- **Con số:** bỏ phiếu cứng **89,60%**, mềm **89,70%**; mô hình riêng tốt nhất **90,10%**.
- **Bức tranh:** bỏ phiếu mềm thường **ngang hoặc nhỉnh hơn** cứng vì giữ được thông tin về độ chắc chắn. Với ba thành viên, cứng chỉ có 3 phiếu nên nhiều ca "hòa 1-1-1" ở bài đa nhóm; ở bài hai nhóm không hòa nhưng vẫn mất thông tin độ chắc.
- **Câu hỏi tiếp:** vì sao ghép không luôn thắng? Do các thành viên có sai giống nhau không?

**Mang về chặng 1:** bỏ phiếu mềm dùng xác suất nên tinh hơn bỏ phiếu cứng; cần xác suất đáng tin.

## Chặng 2: Đa dạng, vì sao ba bản sao không đủ

**Câu hỏi:** ghép ba mô hình **giống nhau** có hơn không?

Ta so hai hội đồng:

- **Hội đồng A (giống nhau):** ba hồi quy logistic chỉ khác `C` (độ nghiêm khắc).
- **Hội đồng B (khác loại):** logistic, rừng, SVM.

Và đo **mức sai trùng nhau**: tỉ lệ ca mà hai mô hình **cùng sai**.

In [5]:
same = VotingClassifier([(f"lr_C{c}", make_pipeline(StandardScaler(), LogisticRegression(C=c, max_iter=2000))) for c in [0.1, 1, 10]], voting="soft")
diverse = soft
res = {}
for name, m in [("A: 3 logistic variants", same), ("B: logistic + forest + svm", diverse)]:
    res[name] = cross_val_score(m, X_rest, y_rest, cv=cv, n_jobs=-1).mean()
print({k: round(float(v), 4) for k, v in res.items()})
errors = {n: (m.fit(a, ya).predict(b) != yb) for n, m in members.items()}
names = list(errors)
overlap = pd.DataFrame([[(errors[i] & errors[j]).mean() / max(errors[i].mean(), 1e-9) for j in names] for i in names], index=names, columns=names)
print("share of model row's errors that the column model ALSO gets wrong:")
print(overlap.round(2))

{'A: 3 logistic variants': 0.8513, 'B: logistic + forest + svm': 0.897}


share of model row's errors that the column model ALSO gets wrong:
          logistic  forest   svm
logistic      1.00    0.54  0.48
forest        0.88    1.00  0.69
svm           0.79    0.69  1.00


**Giải thích từng dòng**

- `[(f"lr_C{c}", ...) for c in [0.1, 1, 10]]`: tạo ba logistic khác nhau chỉ ở `C`. Tên bước tự đặt bằng f-string.
- `errors[n] = m.fit(a, ya).predict(b) != yb`: dãy đúng/sai: mô hình đoán **sai** ở những điểm nào.
- `(errors[i] & errors[j]).mean() / errors[i].mean()`: trong các ca mô hình `i` sai, bao nhiêu phần trăm mô hình `j` cũng sai. 1,0 là sai y hệt; thấp là sai khác nhau.

**Cách đọc.**
- **Con số:** hội đồng A (giống nhau) **85,13%**; hội đồng B (khác loại) **89,70%**.
- **Bức tranh:** bảng chồng lỗi cho thấy hai mô hình khác loại **không sai giống hệt nhau**: mỗi mô hình có một phần lỗi riêng, phần đó mới là chỗ ghép có thể sửa. Ba logistic thì sai gần như cùng chỗ nên ghép chẳng sửa được gì.
- **Quy tắc:** chọn thành viên **khác kiểu** (tuyến tính, cây, láng giềng, SVM), không chọn ba biến thể của một kiểu. Thêm thành viên **kém hơn nhiều** thì thường kéo tụt.

**Mang về chặng 2:** ghép chỉ có lợi khi thành viên sai khác nhau; đo bằng mức lỗi trùng nhau.

## Chặng 3: Stacking, trưởng hội đồng học cách tin ai

**Câu hỏi:** thay vì mỗi người một phiếu ngang nhau, có thể **học** trọng số tin cậy?

Stacking: lấy **dự đoán của các mô hình tầng một** làm **cột đầu vào** cho một mô hình tầng hai (thường là hồi quy logistic đơn giản).

**Cạm bẫy lớn:** nếu mô hình tầng một dự đoán **chính những dòng mà nó đã học** thì dự đoán đó quá đẹp (nó thuộc bài), tầng hai sẽ **tin nhầm**. Vì vậy các dự đoán đưa lên tầng hai phải là dự đoán **ngoài lượt** (*out-of-fold*): mỗi dòng được dự đoán bởi mô hình **chưa thấy nó** (chia kiểu kiểm định chéo).

In [6]:
from sklearn.model_selection import cross_val_predict
oof = np.column_stack([cross_val_predict(m, a, ya, cv=5, method="predict_proba", n_jobs=-1)[:, 1] for m in members.values()])
print("out-of-fold meta-features:", oof.shape, "(rows x 3 models)")
meta = LogisticRegression().fit(oof, ya)
weights = dict(zip(members, meta.coef_[0].round(2)))
print("weights the head of committee learned:", weights)

out-of-fold meta-features: (2250, 3) (rows x 3 models)
weights the head of committee learned: {'logistic': np.float64(0.11), 'forest': np.float64(4.3), 'svm': np.float64(3.36)}


**Giải thích từng dòng**

- `cross_val_predict(m, a, ya, cv=5, method="predict_proba")`: dự đoán xác suất **ngoài lượt**: mỗi dòng nhận xác suất từ mô hình học trên 4/5 dữ liệu **không chứa dòng đó**. (Đã dùng ở C2.)
- `np.column_stack([...])`: xếp ba cột xác suất cạnh nhau thành bảng 3 cột: bảng đầu vào của tầng hai.
- `meta.coef_[0]`: hệ số của hồi quy logistic tầng hai: mô hình nào càng được tin thì hệ số càng lớn.
- `dict(zip(members, ...))`: ghép tên mô hình với hệ số cho dễ đọc.

In [7]:
from sklearn.ensemble import StackingClassifier
stack = StackingClassifier(list(members.items()), final_estimator=LogisticRegression(), cv=5, stack_method="predict_proba", n_jobs=-1)
s_stack = cross_val_score(stack, X_rest, y_rest, cv=cv)
print("stacking CV:", round(s_stack.mean(), 4), "+/-", round(s_stack.std(), 4), "| soft vote:", round(s_soft.mean(), 4), "| best single:", round(first.loc[:2, "CV accuracy"].max(), 4))

stacking CV: 0.9037 +/- 0.0199 | soft vote: 0.897 | best single: 0.901


**Giải thích từng dòng**

- `StackingClassifier(estimators, final_estimator=LogisticRegression(), cv=5)`: làm tất cả bước trên tự động: dự đoán ngoài lượt bằng `cv=5`, rồi huấn luyện tầng hai. `final_estimator` là trưởng hội đồng; nên chọn mô hình **đơn giản** để nó không học vẹt.
- `stack_method="predict_proba"`: đưa xác suất (không phải nhãn cứng) lên tầng hai.
- `cross_val_score(stack, ...)`: đo cả stacking trong kiểm định chéo lồng nhau tự nhiên: mỗi vòng ngoài huấn luyện cả hai tầng chỉ trên phần học (tránh rò rỉ, D3).

**Cách đọc.**
- **Con số:** trọng số tầng hai (một mẫu): {'logistic': np.float64(0.11), 'forest': np.float64(4.3), 'svm': np.float64(3.36)}. Stacking **90,37%** (± 1,99%), bỏ phiếu mềm **89,70%**, mô hình riêng tốt nhất **90,10%**.
- **Bức tranh:** trưởng hội đồng gán **trọng số lớn hơn cho mô hình đáng tin hơn**; stacking thường ngang hoặc nhỉnh hơn bỏ phiếu mềm vì nó tự học trọng số. Chênh lệch **nhỏ hơn độ lệch** thì xem là ngang nhau (D1).
- **Nói thật:** stacking phức tạp hơn (huấn luyện mỗi mô hình 6 lần: 5 ngoài lượt + 1 lần cuối) nhưng lợi ích thường chỉ nửa đến vài điểm phần trăm.

**Mang về chặng 3:** stacking = mô hình tầng hai học từ dự đoán **ngoài lượt** của tầng một; dùng tầng hai đơn giản.

## Chặng 4: Cái giá, thời gian và độ phức tạp

**Câu hỏi:** lợi ích vài phần trăm có đáng cái giá không?

Ta đo **thời gian huấn luyện** và **thời gian dự đoán** của từng cách. Thời gian dự đoán quan trọng nếu mô hình phục vụ **trực tiếp** (ví dụ trả lời trong 50 ms).

In [8]:
import time
rows = []
for name, m in [("logistic", members["logistic"]), ("forest", members["forest"]), ("svm", members["svm"]), ("soft vote", soft), ("stacking", stack)]:
    t0 = time.time(); m.fit(a, ya); t_fit = time.time() - t0
    t0 = time.time(); m.predict(b); t_pred = time.time() - t0
    rows.append({"model": name, "fit s": t_fit, "predict s (for %d rows)" % len(b): t_pred, "test acc": m.score(b, yb)})
cost = pd.DataFrame(rows)
print(cost.round(3).to_string(index=False))

    model  fit s  predict s (for 750 rows)  test acc
 logistic  0.008                     0.001     0.831
   forest  0.892                     0.055     0.897
      svm  0.828                     0.055     0.897
soft vote  1.623                     0.116     0.884
 stacking  6.414                     0.102     0.900


**Giải thích từng dòng**

- `"predict s (for %d rows)" % len(b)`: ghi số dòng vào tên cột; `%d` là chỗ chèn một số nguyên.
- `m.fit(a, ya)`: học lại mỗi mô hình trên `a` để bấm giờ công bằng.

**Cách đọc.**
- **Con số:** huấn luyện: logistic **0,01** s, stacking **6,28** s. Dự đoán cho 750 dòng: logistic **0,001** s, stacking **0,100** s.
- **Bức tranh:** bỏ phiếu và stacking **cộng dồn** chi phí của mọi thành viên (cả lúc dự đoán), và tốn công **bảo trì** (ba mô hình cùng phải được cập nhật). Nếu lợi ích chỉ cỡ độ dao động, chọn **một** mô hình tốt nhất cho gọn.
- **Phán quyết thực tế:** ghép khi (1) cần từng phần trăm (cuộc thi, rủi ro cao), (2) thời gian dự đoán không là vấn đề, (3) có thành viên **khác kiểu thật**.

**Mang về chặng 4:** ghép nhân chi phí vận hành; lợi ích phải xứng với cái giá.

## Chặng 5: Mở tập thi một lần

Ta chọn **hai ứng viên** theo kiểm định chéo (mô hình riêng tốt nhất và stacking), huấn luyện trên **toàn bộ `X_rest`** và chấm **một lần** trên tập thi niêm phong.

In [9]:
best_single_name = first.loc[:2].sort_values("CV accuracy").iloc[-1]["model"]
final_single = members[best_single_name].fit(X_rest, y_rest)
final_stack = stack.fit(X_rest, y_rest)
print("best single (by CV):", best_single_name, "| test:", round(final_single.score(X_test, y_test), 4))
print("stacking              | test:", round(final_stack.score(X_test, y_test), 4), "| test rows:", len(y_test))

best single (by CV): svm | test: 0.899
stacking              | test: 0.911 | test rows: 1000


**Giải thích từng dòng**

- `first.loc[:2].sort_values("CV accuracy").iloc[-1]["model"]`: tên mô hình riêng có điểm kiểm định chéo cao nhất (chọn bằng kiểm định chéo, không bằng tập thi).
- `.fit(X_rest, y_rest)`: học lại trên toàn bộ phần học rồi mới chấm tập thi.

**Cách đọc.**
- **Con số:** mô hình riêng tốt nhất (**svm**): **89,90%**; stacking: **91,10%**. Tập thi có 1000 dòng nên 1 điểm phần trăm là khoảng 10 dòng.
- **Bức tranh:** nếu chênh lệch nhỏ hơn vài điểm phần trăm, báo cáo trung thực là **ngang nhau**, rồi chọn theo **chi phí** (chặng 4), không theo con số cao hơn vài dòng.

**Phán quyết cho chị Hạnh:**
1. Ghép chỉ có ích với thành viên **khác kiểu**; ba bản sao thì vô ích.
2. Bỏ phiếu mềm thường đủ; stacking thêm chút nhưng phức tạp hơn.
3. So bằng **kiểm định chéo kèm độ lệch**, mở tập thi một lần.
4. Ghép cộng dồn chi phí vận hành; đừng ghép nếu lợi ích chỉ cỡ độ dao động.

**Bài học nghề:** trong cuộc thi, ghép hay thắng; trong công việc, **mô hình chạy ổn định trong sản xuất** thường quan trọng hơn thêm nửa điểm.

## Tổng kết

**Mang về theo từng chặng**

| Chặng | Mang về |
|---|---|
| 0 | Bỏ phiếu chỉ đáng khi vượt rõ mô hình riêng tốt nhất; so kèm độ lệch. |
| 1 | Bỏ phiếu mềm dùng xác suất nên tinh hơn bỏ phiếu cứng. |
| 2 | Ghép chỉ có lợi khi thành viên sai khác nhau; đo bằng mức lỗi trùng nhau. |
| 3 | Stacking học trọng số từ dự đoán ngoài lượt; tầng hai nên đơn giản. |
| 4 | Ghép nhân chi phí huấn luyện, dự đoán và bảo trì. |
| 5 | Chênh lệch nhỏ hơn độ dao động thì chọn theo chi phí. |

**Bảng đối chiếu: đời thường, thuật ngữ, tên trong code**

| Đời thường | Thuật ngữ | Tên trong code |
|---|---|---|
| Mỗi bác sĩ một phiếu | bỏ phiếu cứng | `VotingClassifier(voting='hard')` |
| Phiếu có kèm mức chắc chắn | bỏ phiếu mềm | `voting='soft'` |
| Bác sĩ khác chuyên khoa | mô hình đa dạng | logistic + forest + svm |
| Trưởng hội đồng học cách tin ai | stacking | `StackingClassifier` |
| Mô hình tầng hai | bộ ước lượng cuối | `final_estimator` |
| Dự đoán bởi mô hình chưa thấy dòng đó | dự đoán ngoài lượt | `cross_val_predict` |
| Hai mô hình cùng sai một ca | lỗi trùng nhau | `errors[i] & errors[j]` |

**Thử thách tự làm**

1. Thêm `KNeighborsClassifier` (có chuẩn hóa) vào ba thành viên. Lỗi trùng nhau của nó với ba mô hình kia thế nào, và điểm bỏ phiếu mềm có đổi không?
2. Đặt `weights=[1, 2, 1]` trong `VotingClassifier(voting='soft')` để rừng có trọng số 2. Điểm đổi ra sao?
3. Dùng `HistGradientBoostingClassifier` thay rừng. Stacking còn lợi không?
4. Thử `final_estimator=RandomForestClassifier()` cho stacking. Có quá khớp không? So điểm kiểm định chéo và điểm thi.

**Checklist năng lực** (tự tick khi làm được)

- [ ] Dựng được bỏ phiếu cứng và mềm.
- [ ] Đo mức lỗi trùng nhau giữa hai mô hình.
- [ ] Giải thích vì sao stacking cần dự đoán ngoài lượt.
- [ ] Nói được khi nào ghép không đáng.
- [ ] So ghép và mô hình đơn bằng kiểm định chéo kèm độ lệch.

## Phần thưởng (Tip pro): Blending và cái bẫy rò rỉ trong stacking


Một biến thể gọi là **blending**: thay vì dự đoán ngoài lượt bằng kiểm định chéo, ta để riêng một phần dữ liệu (ví dụ 20%) làm **tập blend**, tầng một học trên 80%, tầng hai học trên dự đoán của tầng một cho 20% đó. Đơn giản hơn nhưng lãng phí dữ liệu. Lỗi hay gặp nhất khi tự dựng stacking: **cho tầng hai học trên dự đoán của tầng một ở chính tập tầng một đã học**, điểm sẽ đẹp giả tạo (cùng họ rò rỉ ở D3). `StackingClassifier` đã làm đúng nên nên dùng bản có sẵn.